In [30]:
from sklearn.cluster import DBSCAN
import json
from transformers import BertModel, BertTokenizerFast
import torch
from collections import defaultdict

In [31]:
# load the sentences and correct labels
all_sentences = []

with open("../01_data/annotations/annotations_labelstudio_export.json", "r") as f:
    data = json.load(f)

for sentence in data:
    text = sentence["data"]["sentence"]
    labels = []
    results = sentence["annotations"][0]["result"]
    labels = [r["value"]["text"] for r in results]
    spans = [
        {
            "start": r["value"]["start"],
            "end": r["value"]["end"]
        }
        for r in results if r["type"] == "labels"
    ]

    all_sentences.append({
        "text": text,
        "labels": labels,
        "spans": spans
    })

In [32]:
# extract all group mentions present
group_mentions_present = [ex for ex in all_sentences if ex["labels"]]
len(group_mentions_present)

2588

In [33]:
# define tokenizer
tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

# download model with pretrained weights 
model = BertModel.from_pretrained("bert-base-uncased", output_hidden_states=True) 

# set model to evaluation mode
model.eval()

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False

In [23]:
# define function that returns the word embeddings for all group mentions within sentence
def extract_embedding(tokenizer, model, sentence, labels):
    encoding = tokenizer(sentence, return_tensors="pt", return_offsets_mapping=True, truncation=True)
    offset_mapping = encoding.pop("offset_mapping")[0].tolist()

    with torch.no_grad():
        outputs = model(**encoding)
        hidden_states = outputs.last_hidden_state[0]

    mention_embeddings = []

    for mention in labels:
        mention_start = int(mention["start"])
        mention_end = int(mention["end"])
        token_indices = [
            i for i, (start, end) in enumerate(offset_mapping)
            if start >= mention_start and end <= mention_end
        ]

        if token_indices:
            mention_embedding = hidden_states[torch.tensor(token_indices)].mean(dim=0)
            mention_embeddings.append(mention_embedding)

    return mention_embeddings

In [34]:
# define a function that returns the word embeddings only for the group mention span
def extract_span_embedding(tokenizer, model, labels):
    mention_embeddings = []
    for mention in labels:
        encoding = tokenizer(mention, return_tensors="pt", truncation=True)
        with torch.no_grad():
            outputs = model(**encoding)
            hidden_states = outputs.last_hidden_state[0]
            mention_embedding = hidden_states.mean(dim=0)
            mention_embeddings.append(mention_embedding)
    return mention_embeddings

In [35]:
# get embeddings for all mentions
all_embeddings = []
for sentence in group_mentions_present:
    text = sentence["text"]
    labels = sentence["labels"]
    spans = sentence["spans"]
    #mention_embeddings = extract_embedding(tokenizer, model, text, spans)
    mention_embeddings = extract_span_embedding(tokenizer, model, labels)
    all_embeddings.append(mention_embeddings)

# flatten the embeddings list
flat_embeddings = [embedding for sentence_mentions in all_embeddings for embedding in sentence_mentions]

In [36]:
X = torch.stack(flat_embeddings).cpu().numpy()
dbscan = DBSCAN(eps=.1, min_samples=3, metric='cosine') 
labels = dbscan.fit_predict(X)

In [37]:
# get all mentions
all_mentions = [mention for sentence in group_mentions_present for mention in sentence["labels"]]

# pair them with their labels
mentions_labels = zip(all_mentions, labels)

In [38]:
cluster_dict = defaultdict(list)

for mention, label in zip(all_mentions, labels):
    cluster_dict[label].append(mention)

for cluster_id, mentions in cluster_dict.items():
    print(f"Cluster {cluster_id} ({len(mentions)} mentions):")
    for mention in mentions:
        print(f"  - {mention}")
    print()

Cluster 0 (9 mentions):
  - judges
  - Judges
  - judges
  - judges
  - judges
  - judges
  - judges
  - judges
  - judges

Cluster -1 (1327 mentions):
  - those aged under 18
  - senior British bankers
  - the most disadvantaged
  - neighbours
  - the families of those who were lost
  - the entrepreneurs of the future
  - clinicians-GPs
  - predatory individuals
  - officers in training
  - persecuted Christians around the world
  - unpaid family carers
  - business leaders from the sector
  - middle-grade emergency medicine doctors
  - child sex slaves
  - post-graduate trainee teachers
  - those who fail in their responsibilities
  - employers of apprentices
  - a judge
  - young people under 25
  - British citizen
  - children born to British unmarried fathers
  - any child under the age of 18
  - School Teachers
  - the poorest 20% of families in this country
  - unaccompanied children from Calais
  - Bristol pupils
  - those with hidden disabilities
  - parents who refuse to pay
